In [1]:
# Sel ini menghasilkan TIGA dataset cabang (kota) terpisah untuk Tugas Mandiri Pertemuan 3
import numpy as np
import pandas as pd

kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-08-01", "2026-08-31", freq="D")

cabang_kota = {"Magelang": 101, "Yogyakarta": 202, "Semarang": 303}

for kota, seed in cabang_kota.items():
    np.random.seed(seed)  # seed berbeda tiap kota agar datanya bervariasi, namun tetap konsisten/reproducible
    n = 200
    data_cabang = {
        "order_id": [f"{kota[:3].upper()}-{2000 + i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "unit_terjual": np.random.randint(1, 8, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    }
    df_cabang = pd.DataFrame(data_cabang)
    df_cabang["kota"] = kota
    nama_file = f"transaksi_{kota.lower()}.csv"
    df_cabang.to_csv(nama_file, index=False)
    print(f"Berkas '{nama_file}' berhasil dibuat: {df_cabang.shape[0]} baris")

print("\nKetiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.")

Berkas 'transaksi_magelang.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_yogyakarta.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_semarang.csv' berhasil dibuat: 200 baris

Ketiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.


In [2]:
!hdfs dfs -mkdir -p /tarin/ecommerce/raw
!hdfs dfs -mkdir -p /tarin/ecommerce/processed
!hdfs dfs -ls /tarin/ecommerce

Found 2 items
drwxr-xr-x   - tarin supergroup          0 2026-09-09 14:21 /tarin/ecommerce/processed
drwxr-xr-x   - tarin supergroup          0 2026-09-09 14:21 /tarin/ecommerce/raw


In [3]:
# Upload (put) berkas lokal ke dalam HDFS
!hdfs dfs -put transaksi_magelang.csv transaksi_yogyakarta.csv transaksi_semarang.csv /tarin//ecommerce/raw

# Verifikasi berkas sudah ada di HDFS
!hdfs dfs -ls /tarin/ecommerce/raw

Found 3 items
-rw-r--r--   1 tarin supergroup      12329 2026-09-09 14:30 /tarin/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 tarin supergroup      12154 2026-09-09 14:30 /tarin/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 tarin supergroup      12684 2026-09-09 14:30 /tarin/ecommerce/raw/transaksi_yogyakarta.csv


In [4]:
!hdfs dfs -get /tarin/ecommerce/raw/transaksi_magelang.csv magelang_dari_hdfs.csv
!hdfs dfs -get /tarin/ecommerce/raw/transaksi_semarang.csv semarang_dari_hdfs.csv
!hdfs dfs -get /tarin/ecommerce/raw/transaksi_yogyakarta.csv yogyakarta_dari_hdfs.csv

In [5]:
import pandas as pd
df_magelang=pd.read_csv("magelang_dari_hdfs.csv")
df_semarang=pd.read_csv("semarang_dari_hdfs.csv")
df_yogyakarta=pd.read_csv("yogyakarta_dari_hdfs.csv")

df_gabungan=pd.concat([df_magelang,df_semarang,df_yogyakarta], ignore_index=True)
df_gabungan

,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang
...,...,...,...,...,...,...,...
595,YOG-2195,2026-08-25,Rumah Tangga,4,250000,Kartu Kredit,Yogyakarta
596,YOG-2196,2026-08-18,Fashion,1,150000,COD,Yogyakarta
597,YOG-2197,2026-08-27,Makanan & Minuman,4,25000,Kartu Kredit,Yogyakarta
598,YOG-2198,2026-08-07,Rumah Tangga,1,25000,COD,Yogyakarta


In [8]:
df_gabungan["kota"].value_counts()

kota
Magelang      200
Semarang      200
Yogyakarta    200
Name: count, dtype: int64

In [9]:
df_gabungan["total_pendapatan"] = df_gabungan["unit_terjual"] * df_gabungan["harga_satuan"]
df_gabungan.head()

,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota,total_pendapatan
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang,50000
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang,175000
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang,175000
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang,150000
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang,700000


In [10]:
ringkasan_kota_kategori = df_gabungan.groupby(["kota","kategori"])["total_pendapatan"].sum().reset_index()
ringkasan_kota_kategori

,kota,kategori,total_pendapatan
0,Magelang,Elektronik,18775000
1,Magelang,Fashion,27750000
2,Magelang,Kesehatan & Kecantikan,17375000
3,Magelang,Makanan & Minuman,17525000
4,Magelang,Rumah Tangga,12200000
5,Semarang,Elektronik,21425000
6,Semarang,Fashion,26425000
7,Semarang,Kesehatan & Kecantikan,13525000
8,Semarang,Makanan & Minuman,19750000
9,Semarang,Rumah Tangga,10975000


In [11]:
df_gabungan.to_csv("data_gabungan_bersih.csv", index=False)
ringkasan_kota_kategori.to_csv("ringkasan_kota_kategori.csv", index=False)
print("Kedua berkas berhasil disimpan didisk lokal.")

Kedua berkas berhasil disimpan didisk lokal.


In [12]:
!hdfs dfs -put data_gabungan_bersih.csv /tarin/ecommerce/processed
!hdfs dfs -put ringkasan_kota_kategori.csv /tarin/ecommerce/processed

!hdfs dfs -ls /tarin/ecommerce/processed

Found 2 items
-rw-r--r--   1 tarin supergroup      41257 2026-09-09 15:03 /tarin/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 tarin supergroup        530 2026-09-09 15:03 /tarin/ecommerce/processed/ringkasan_kota_kategori.csv
